# Attribution Graphs

Build directed feature-to-feature computation graphs using SAE features
as nodes and activation-based attribution as edges. Based on Anthropic's
2025 circuit tracing methodology.

This notebook covers the `irtk.attribution_graphs` module.

## Why This Matters

Attribution graphs trace feature-to-feature computation across layers, revealing the directed graph of how information flows from input features through intermediate computations to output features. This provides a circuit-level view of model computation with explicit paths and interaction strengths.

**Key references:**
- [Conmy et al. (2023) "Towards Automated Circuit Discovery"](https://arxiv.org/abs/2304.14997) — Automated methods for circuit finding via edge pruning
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import attribution_graphs, sae

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## 1. Train SAEs and Build Graph

Train SAEs at multiple layers, then build an attribution graph.

In [ ]:
prompts = [
    "The Eiffel Tower is located in",
    "The capital of France is",
    "London is the capital of",
    "Berlin is a city in",
]
token_seqs = [model.to_tokens(p) for p in prompts]

# Train SAEs at two layers
saes = {}
for layer in [4, 8]:
    hook = f"blocks.{layer}.hook_resid_post"
    result = sae.train_sae(model, hook, token_seqs, n_features=64, n_epochs=5, lr=1e-3)
    saes[hook] = result.sae
    print(f"Layer {layer}: trained SAE")

# Build attribution graph
paris_id = tokenizer.encode(" Paris")[0]
def metric(logits):
    return float(logits[-1, paris_id])

tokens = model.to_tokens("The Eiffel Tower is located in")
graph = attribution_graphs.build_attribution_graph(model, saes, tokens, metric)
print(f"Graph: {len(graph.nodes)} nodes, {len(graph.edges)} edges")

## 2. Node Importance

Which features are most important in the computation?

In [ ]:
result = attribution_graphs.node_importance(graph, method="flow")

print("Top 10 most important features:")
for node_idx, imp in result['top_nodes'][:10]:
    print(f"  {graph.node_labels[node_idx]}: importance = {imp:.4f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(min(20, len(result['top_nodes']))),
       [imp for _, imp in result['top_nodes'][:20]], color='steelblue')
ax.set_xlabel('Feature Rank')
ax.set_ylabel('Importance')
ax.set_title('Top Feature Importances')
plt.tight_layout()
plt.show()

## 3. Prune and Visualize

Prune to essential subgraph and prepare visualization data.

In [ ]:
pruned = attribution_graphs.prune_graph(graph, threshold=0.3, max_nodes=15)
print(f"Pruned graph: {len(pruned.nodes)} nodes, {len(pruned.edges)} edges")

vis = attribution_graphs.visualize_attribution_graph(pruned, top_k_nodes=15)
print(f"\nVisualization: {vis['n_nodes']} nodes, {vis['n_edges']} edges")
for node in vis['nodes'][:5]:
    print(f"  {node['label']}: importance = {node['importance']:.4f}")

## 4. Faithfulness Evaluation

Does the pruned graph reproduce the full model's behavior?

In [ ]:
result = attribution_graphs.attribution_graph_faithfulness(
    model, pruned, saes, tokens, metric
)

print(f"Full metric: {result['full_metric']:.4f}")
print(f"Graph metric: {result['graph_metric']:.4f}")
print(f"Faithfulness: {result['faithfulness']:.2%}")
print(f"Feature coverage: {result['feature_coverage']:.2%}")

## Summary

| Function | Purpose |
|----------|--------|
| `build_attribution_graph()` | Construct feature-to-feature computation graph |
| `node_importance()` | Score feature importance via flow or degree |
| `prune_graph()` | Extract minimal faithful subgraph |
| `visualize_attribution_graph()` | Prepare graph data for rendering |
| `attribution_graph_faithfulness()` | Evaluate pruned graph's fidelity |